In [ ]:
print('bronze_started')

In [ ]:
from pyspark.sql import functions as f
import uuid

In [ ]:
%sql
use catalog marketing_analytics_capstone;
USE schema bronze;

In [ ]:
# BASE PATH 
BASE_PATH = "s3://marketing-analytics-capstone/"  

# Bronze output path
BRONZE_PATH = "s3://marketing-analytics-capstone/bronze/"  

# Checkpoints path
CHECKPOINTS_PATH = "s3://marketing-analytics-capstone/checkpoints/bronze"   

In [ ]:

df_bronze_campaigns = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")                   
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINTS_PATH}/campaigns/schema/")
    .load(f"{BASE_PATH}raw/")                
)

df_bronze_campaigns = (
    df_bronze_campaigns
    .withColumn("ingestion_timestamp", f.current_timestamp())
    .withColumn("source_file_name",    f.lit("campaigns"))
    .withColumn("batch_id",            f.lit(str(uuid.uuid4())))
    .withColumn(
        "source_month",
        f.date_format(f.to_date(f.col("Date"), "yyyy-MM-dd"), "yyyy-MM")
    )
)

query_campaigns = (
    df_bronze_campaigns
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINTS_PATH}/campaigns/checkpoints/")
    .option("mergeSchema", "true")
    .partitionBy("source_month")
    .trigger(availableNow=True)
    .toTable("marketing_analytics_capstone.bronze.bronze_campaigns")
)

query_campaigns.awaitTermination()
print("✅ bronze_campaigns loaded")

In [ ]:
(
    spark.read.table("marketing_analytics_capstone.bronze.bronze_campaigns")
    .groupBy("source_month", "batch_id")
    .count()
    .orderBy("source_month")
    .show(truncate=False)
)

In [ ]:
%sql
SELECT
    source_month,
    COUNT(*)              AS total_records,
    COUNT(DISTINCT batch_id) AS total_batches,
    MIN(ingestion_timestamp) AS first_loaded,
    MAX(ingestion_timestamp) AS last_loaded
FROM marketing_analytics_capstone.bronze.bronze_campaigns
GROUP BY source_month
ORDER BY source_month;